# Day 10: LangGraph Wrap-up + n8n Workflow Automation

In [ ]:
!pip install -q langchain langchain-groq langgraph langgraph-checkpoint-sqlite

In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

# 🗄️ a real file on disk — survives restarts
conn = sqlite3.connect("checkpoints.sqlite", check_same_thread=False)
memory = SqliteSaver(conn)

graph = builder.compile(checkpointer=memory)   # 👈 the only change

In [ ]:
config = {"configurable": {"thread_id": "session-1"}}

# ▶️ first run (maybe the program stops here)
graph.invoke({"question": "Draft a plan for a science fair"}, config)

# ...restart Python entirely, reconnect the SAME file...
conn = sqlite3.connect("checkpoints.sqlite", check_same_thread=False)
graph = builder.compile(checkpointer=SqliteSaver(conn))

# 🔎 the state is still there
snapshot = graph.get_state(config)
print("Resumed state:", snapshot.values)

In [ ]:
# 🐾 every checkpoint, newest first
for snap in graph.get_state_history(config):
    step = snap.metadata.get("step")
    nxt  = snap.next            # which node runs next
    print(f"step {step} · next={nxt} · keys={list(snap.values)}")

In [ ]:
history = list(graph.get_state_history(config))
old = history[2]                              # some earlier checkpoint
graph.invoke(None, old.config)                # ⏪ replay from there

## What is n8n, really?

**BASH snippet**

```bash
npm install n8n -g
n8n
```
**TEXT snippet**

```text
Editor is now accessible via:
http://localhost:5678/
```
**BASH snippet**

```bash
npx n8n                  # just start it
npx n8n@latest           # force the newest version
npx n8n start --tunnel   # start with a public webhook URL (see below)
```
**BASH snippet**

```bash
npx n8n start --tunnel
```
**BASH snippet**

```bash
  docker run -it --rm -p 5678:5678 -v ~/.n8n:/home/node/.n8n docker.n8n.io/n8nio/n8n
  
```

## The n8n mental model vs LangGraph

In [ ]:
   Answer this clearly: {{ $json.body.question }}
   

In [ ]:
   {{ $json.text }}
   

**BASH snippet**

```bash
   curl -X POST "<YOUR_TEST_WEBHOOK_URL>" \
     -H "Content-Type: application/json" \
     -d '{"question": "What is n8n in one sentence?"}'
   
```

In [ ]:
   Classify this email as exactly one word — urgent, normal, or spam:
   Subject: {{ $json.subject }}
   Body: {{ $json.text }}
   

In [ ]:
# 💾 LANGGRAPH: PERSIST TO DISK
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
conn = sqlite3.connect("checkpoints.sqlite", check_same_thread=False)
graph = builder.compile(checkpointer=SqliteSaver(conn))

config = {"configurable": {"thread_id": "session-1"}}
graph.invoke(inputs, config)                 # runs + auto-saves
graph.get_state(config)                       # 🔎 current state
list(graph.get_state_history(config))         # 🐾 full trace